# Bayesian Walk Fine-Tuning

Fine-tunes the Bayesian Walk model (stick-or-switch with softmax best response)
on both aggregate and switch-stage metrics.

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

In [ ]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

## 2. Model Definition

In [ ]:
def make_bayesian_walk(tau_prior=5.6, tau_softmax=0.1, epsilon=0.2, epsilon_switch=0.3):
    """Bayesian Walk: stick-or-switch with softmax best response on switch."""
    def predict_fn(record):
        env = record['env_config']
        values = env['values']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            switch_dist = [softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax)
                           for i in range(3)]

            per_agent = []
            for i in range(3):
                if prev_roles is None:
                    per_agent.append(switch_dist[i])
                else:
                    stick = np.zeros(3)
                    stick[prev_roles[i]] = 1.0
                    mixed = (1.0 - epsilon_switch) * stick + epsilon_switch * switch_dist[i]
                    per_agent.append(mixed)

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


def precompute_trajectories(records, tau_prior, epsilon):
    """Precompute posterior + game state per stage for each record.

    Depends only on (tau_prior, epsilon) and observed human actions.
    Returns list of trajectories (one per record), each a list of stage dicts:
        prior, intent, thp, ehp, human_combo, prev_roles.
    """
    trajectories = []
    for record in records:
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        turn_idx = 0
        prev_roles = None
        stages = []

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            stages.append({
                'prior': prior.copy(),
                'intent': intent,
                'thp': thp,
                'ehp': ehp,
                'human_combo': human_combo,
                'prev_roles': prev_roles,
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = list(human_roles)
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent_t = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent_t, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent_t, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent_t, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        trajectories.append(stages)
    return trajectories


def predict_from_trajectory(record, trajectory, values, tau_softmax, epsilon_switch):
    """Generate predictions from precomputed trajectory (no Bayesian updates needed)."""
    results = []
    for stage in trajectory:
        prior = stage['prior']
        intent, thp, ehp = stage['intent'], stage['thp'], stage['ehp']
        prev_roles = stage['prev_roles']
        human_combo = stage['human_combo']

        switch_dist = [softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax)
                       for i in range(3)]

        per_agent = []
        for i in range(3):
            if prev_roles is None:
                per_agent.append(switch_dist[i])
            else:
                stick = np.zeros(3)
                stick[prev_roles[i]] = 1.0
                mixed = (1.0 - epsilon_switch) * stick + epsilon_switch * switch_dist[i]
                per_agent.append(mixed)

        predicted_dist = {}
        for r0 in range(3):
            for r1 in range(3):
                for r2 in range(3):
                    combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                    predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

        results.append({
            'predicted_dist': predicted_dist,
            'human_combo': human_combo,
            'model_marginal': np.mean(per_agent, axis=0),
        })
    return results

## 3. Aggregate Fine-Tuning

3-stage: coarse grid -> refined grid -> scipy polishing.

In [ ]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])

metric = 'combo_r'  # optimize combo Pearson r

def evaluate_walk(records, tau_prior, tau_softmax, epsilon, epsilon_switch):
    """Run Bayesian Walk at given params (uncached, for scipy)."""
    results = run_predictions(records, make_bayesian_walk(
        tau_prior=tau_prior, tau_softmax=tau_softmax,
        epsilon=epsilon, epsilon_switch=epsilon_switch))
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
        'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_from_cache(records, trajectories, tau_softmax, epsilon_switch):
    """Evaluate using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(record, trajectories[idx],
                                       record['env_config']['values'],
                                       tau_softmax, epsilon_switch)
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_softmax': tau_softmax, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_walk(records, tau_prior_vals, tau_softmax_vals, eps_vals, eps_switch_vals):
    """Cached grid search: precompute per (tau_prior, eps), sweep (tau_softmax, eps_switch)."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(tau_softmax_vals, eps_switch_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for ts, eps_sw in inner_combos:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_from_cache(records, trajectories, ts, eps_sw)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results


In [ ]:
# Grid ranges (matching online_model_analysis enhanced.ipynb)
tau_prior_min,   tau_prior_max,   tau_prior_steps   = 0.1, 20.0, 10
tau_softmax_min, tau_softmax_max, tau_softmax_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.001, 0.2, 10
# epsilon_switch: new parameter, search broadly
eps_switch_vals = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0])

tau_prior_vals   = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
tau_softmax_vals = np.linspace(tau_softmax_min, tau_softmax_max, tau_softmax_steps)
eps_vals         = np.geomspace(eps_min, eps_max, eps_steps)

total = len(tau_prior_vals) * len(tau_softmax_vals) * len(eps_vals) * len(eps_switch_vals)
print(f'Coarse grid: {len(tau_prior_vals)} x {len(tau_softmax_vals)} x {len(eps_vals)} x {len(eps_switch_vals)} = {total} points')
print(f'  tau_prior:    linspace({tau_prior_min}, {tau_prior_max}, {tau_prior_steps})')
print(f'  tau_softmax:  linspace({tau_softmax_min}, {tau_softmax_max}, {tau_softmax_steps})')
print(f'  epsilon:      geomspace({eps_min}, {eps_max}, {eps_steps})')
print(f'  eps_switch:   {eps_switch_vals}')
print()

sweep_results = grid_search_walk(records, tau_prior_vals, tau_softmax_vals, eps_vals, eps_switch_vals)
best = pick_best(sweep_results, metric)
print(f'\nCoarse best: tau_prior={best["tau_prior"]:.4f} tau_softmax={best["tau_softmax"]:.4f} '
      f'eps={best["epsilon"]:.6f} eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

### Step 2: Refined Grid

In [ ]:
# Refine around coarse best (matching online_model_analysis enhanced.ipynb)
tp_step  = (tau_prior_max - tau_prior_min) / tau_prior_steps
ts_step  = (tau_softmax_max - tau_softmax_min) / tau_softmax_steps
eps_ratio = (eps_max / eps_min) ** (1.0 / eps_steps)

fine_tp = np.linspace(
    max(tau_prior_min, best['tau_prior'] - tp_step),
    min(tau_prior_max, best['tau_prior'] + tp_step),
    10,
)
fine_ts = np.linspace(
    max(tau_softmax_min, best['tau_softmax'] - ts_step),
    min(tau_softmax_max, best['tau_softmax'] + ts_step),
    10,
)
fine_eps = np.geomspace(
    max(eps_min, best['epsilon'] / eps_ratio),
    min(eps_max, best['epsilon'] * eps_ratio),
    10,
)
# Refine eps_switch around best
def refine_range(val, vals_arr, n=8):
    sorted_vals = np.sort(vals_arr)
    idx = np.searchsorted(sorted_vals, val)
    lo = sorted_vals[max(0, idx - 1)]
    hi = sorted_vals[min(len(sorted_vals) - 1, idx + 1)] if idx < len(sorted_vals) - 1 else sorted_vals[-1]
    return np.linspace(lo, hi, n)

fine_esw = refine_range(best['epsilon_switch'], eps_switch_vals)

total_fine = len(fine_tp) * len(fine_ts) * len(fine_eps) * len(fine_esw)
print(f'Refined grid: {len(fine_tp)} x {len(fine_ts)} x {len(fine_eps)} x {len(fine_esw)} = {total_fine} points')
print(f'  tau_prior:   [{fine_tp[0]:.4f}, {fine_tp[-1]:.4f}]')
print(f'  tau_softmax: [{fine_ts[0]:.4f}, {fine_ts[-1]:.4f}]')
print(f'  epsilon:     [{fine_eps[0]:.6f}, {fine_eps[-1]:.6f}]')
print(f'  eps_switch:  [{fine_esw[0]:.4f}, {fine_esw[-1]:.4f}]')
print()

refined_results = grid_search_walk(records, fine_tp, fine_ts, fine_eps, fine_esw)
sweep_results.extend(refined_results)
best = pick_best(sweep_results, metric)
print(f'\nRefined best: tau_prior={best["tau_prior"]:.4f} tau_softmax={best["tau_softmax"]:.4f} '
      f'eps={best["epsilon"]:.6f} eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

### Step 3: Scipy Polishing

In [ ]:
# Scipy L-BFGS-B polishing around refined best
def objective(params):
    tp, ts, eps, eps_sw = params
    res = evaluate_walk(records, tp, ts, eps, eps_sw)
    return -res[metric]

tp_lo = max(tau_prior_min, best['tau_prior'] - tp_step / 2)
tp_hi = min(tau_prior_max, best['tau_prior'] + tp_step / 2)
ts_lo = max(tau_softmax_min, best['tau_softmax'] - ts_step / 2)
ts_hi = min(tau_softmax_max, best['tau_softmax'] + ts_step / 2)
eps_lo = max(1e-6, best['epsilon'] / 2)
eps_hi = min(eps_max, best['epsilon'] * 2)
esw_lo = max(0.01, best['epsilon_switch'] - 0.1)
esw_hi = min(1.0, best['epsilon_switch'] + 0.1)

x0 = [best['tau_prior'], best['tau_softmax'], best['epsilon'], best['epsilon_switch']]
bounds = [(tp_lo, tp_hi), (ts_lo, ts_hi), (eps_lo, eps_hi), (esw_lo, esw_hi)]

print(f'Scipy optimization (metric={metric}) ...')
print(f'  bounds: tau_prior=[{tp_lo:.4f}, {tp_hi:.4f}], tau_softmax=[{ts_lo:.4f}, {ts_hi:.4f}]')
print(f'          epsilon=[{eps_lo:.6f}, {eps_hi:.6f}], eps_switch=[{esw_lo:.4f}, {esw_hi:.4f}]')

opt_result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                      options={'maxiter': 50, 'ftol': 1e-6})
opt_tp, opt_ts, opt_eps, opt_esw = opt_result.x
print(f'  Optimal: tau_prior={opt_tp:.4f} tau_softmax={opt_ts:.4f} eps={opt_eps:.6f} eps_switch={opt_esw:.4f}')
print(f'  objective={-opt_result.fun:.4f}')

opt_eval = evaluate_walk(records, opt_tp, opt_ts, opt_eps, opt_esw)
sweep_results.append(opt_eval)
best = pick_best(sweep_results, metric)
print(f'\nFinal best: tau_prior={best["tau_prior"]:.4f} tau_softmax={best["tau_softmax"]:.4f} '
      f'eps={best["epsilon"]:.6f} eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

## 4. Switch-Stage Fine-Tuning

In [ ]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered


def evaluate_walk_switch(records, tau_prior, tau_softmax, epsilon, epsilon_switch):
    """Run on full history, compute metrics only on switch stages (uncached, for scipy)."""
    full_results = run_predictions(records, make_bayesian_walk(
        tau_prior=tau_prior, tau_softmax=tau_softmax,
        epsilon=epsilon, epsilon_switch=epsilon_switch))
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
                'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
        'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_switch_from_cache(records, trajectories, tau_softmax, epsilon_switch):
    """Evaluate switch-stage metrics using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(record, trajectories[idx],
                                       record['env_config']['values'],
                                       tau_softmax, epsilon_switch)
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_softmax': tau_softmax, 'epsilon_switch': epsilon_switch,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_softmax': tau_softmax, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_walk_switch(records, tau_prior_vals, tau_softmax_vals, eps_vals, eps_switch_vals):
    """Cached grid search optimizing switch-stage correlation."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(tau_softmax_vals, eps_switch_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for ts, eps_sw in inner_combos:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_switch_from_cache(records, trajectories, ts, eps_sw)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

In [ ]:
# Coarse grid (same ranges as aggregate tuning)
print('=== Switch-Stage Fine-Tuning: Coarse Grid ===')
sw_total = len(tau_prior_vals) * len(tau_softmax_vals) * len(eps_vals) * len(eps_switch_vals)
print(f'{sw_total} points\n')

sw_sweep = grid_search_walk_switch(records, tau_prior_vals, tau_softmax_vals, eps_vals, eps_switch_vals)
best_sw = pick_best(sw_sweep, metric)
print(f'\nCoarse best: tau_prior={best_sw["tau_prior"]:.2f} tau_softmax={best_sw["tau_softmax"]:.2f} '
      f'eps={best_sw["epsilon"]:.3f} eps_switch={best_sw["epsilon_switch"]:.2f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

In [ ]:
# Refined grid around switch-stage coarse best
fine_sw_tp = np.linspace(
    max(tau_prior_min, best_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_sw['tau_prior'] + tp_step),
    10,
)
fine_sw_ts = np.linspace(
    max(tau_softmax_min, best_sw['tau_softmax'] - ts_step),
    min(tau_softmax_max, best_sw['tau_softmax'] + ts_step),
    10,
)
fine_sw_eps = np.geomspace(
    max(eps_min, best_sw['epsilon'] / eps_ratio),
    min(eps_max, best_sw['epsilon'] * eps_ratio),
    10,
)
fine_sw_esw = refine_range(best_sw['epsilon_switch'], eps_switch_vals)

total_sw_fine = len(fine_sw_tp) * len(fine_sw_ts) * len(fine_sw_eps) * len(fine_sw_esw)
print(f'Refined grid (switch-stage): {total_sw_fine} points')
print(f'  tau_prior:   [{fine_sw_tp[0]:.4f}, {fine_sw_tp[-1]:.4f}]')
print(f'  tau_softmax: [{fine_sw_ts[0]:.4f}, {fine_sw_ts[-1]:.4f}]')
print(f'  epsilon:     [{fine_sw_eps[0]:.6f}, {fine_sw_eps[-1]:.6f}]')
print(f'  eps_switch:  [{fine_sw_esw[0]:.4f}, {fine_sw_esw[-1]:.4f}]')
print()

sw_refined = grid_search_walk_switch(records, fine_sw_tp, fine_sw_ts, fine_sw_eps, fine_sw_esw)
sw_sweep.extend(sw_refined)
best_sw = pick_best(sw_sweep, metric)
print(f'\nRefined best: tau_prior={best_sw["tau_prior"]:.4f} tau_softmax={best_sw["tau_softmax"]:.4f} '
      f'eps={best_sw["epsilon"]:.6f} eps_switch={best_sw["epsilon_switch"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

In [ ]:
# Scipy polishing for switch-stage tuning
def objective_sw(params):
    tp, ts, eps, eps_sw = params
    res = evaluate_walk_switch(records, tp, ts, eps, eps_sw)
    return -res[metric]

sw_tp_lo = max(tau_prior_min, best_sw['tau_prior'] - tp_step / 2)
sw_tp_hi = min(tau_prior_max, best_sw['tau_prior'] + tp_step / 2)
sw_ts_lo = max(tau_softmax_min, best_sw['tau_softmax'] - ts_step / 2)
sw_ts_hi = min(tau_softmax_max, best_sw['tau_softmax'] + ts_step / 2)
sw_eps_lo = max(1e-6, best_sw['epsilon'] / 2)
sw_eps_hi = min(eps_max, best_sw['epsilon'] * 2)
sw_esw_lo = max(0.01, best_sw['epsilon_switch'] - 0.1)
sw_esw_hi = min(1.0, best_sw['epsilon_switch'] + 0.1)

sw_x0 = [best_sw['tau_prior'], best_sw['tau_softmax'], best_sw['epsilon'], best_sw['epsilon_switch']]
sw_bounds = [(sw_tp_lo, sw_tp_hi), (sw_ts_lo, sw_ts_hi), (sw_eps_lo, sw_eps_hi), (sw_esw_lo, sw_esw_hi)]

print('Scipy optimization (switch-stage) ...')
print(f'  bounds: tau_prior=[{sw_tp_lo:.4f}, {sw_tp_hi:.4f}], tau_softmax=[{sw_ts_lo:.4f}, {sw_ts_hi:.4f}]')
print(f'          epsilon=[{sw_eps_lo:.6f}, {sw_eps_hi:.6f}], eps_switch=[{sw_esw_lo:.4f}, {sw_esw_hi:.4f}]')

sw_opt = minimize(objective_sw, sw_x0, method='L-BFGS-B', bounds=sw_bounds,
                  options={'maxiter': 50, 'ftol': 1e-6})
sw_opt_tp, sw_opt_ts, sw_opt_eps, sw_opt_esw = sw_opt.x
print(f'  Optimal: tau_prior={sw_opt_tp:.4f} tau_softmax={sw_opt_ts:.4f} '
      f'eps={sw_opt_eps:.6f} eps_switch={sw_opt_esw:.4f}')
print(f'  objective={-sw_opt.fun:.4f}')

sw_opt_eval = evaluate_walk_switch(records, sw_opt_tp, sw_opt_ts, sw_opt_eps, sw_opt_esw)
sw_sweep.append(sw_opt_eval)
best_sw = pick_best(sw_sweep, metric)
print(f'\nFinal best (switch-stage): tau_prior={best_sw["tau_prior"]:.4f} tau_softmax={best_sw["tau_softmax"]:.4f} '
      f'eps={best_sw["epsilon"]:.6f} eps_switch={best_sw["epsilon_switch"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

## 5. Save Parameters

In [ ]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'tau_prior': best['tau_prior'], 'tau_softmax': best['tau_softmax'],
        'epsilon': best['epsilon'], 'epsilon_switch': best['epsilon_switch'],
        'combo_r': best['combo_r'], 'marg_r': best['marg_r'], 'mean_ll': best['mean_ll'],
    },
    'switch_stage_tuned': {
        'tau_prior': best_sw['tau_prior'], 'tau_softmax': best_sw['tau_softmax'],
        'epsilon': best_sw['epsilon'], 'epsilon_switch': best_sw['epsilon_switch'],
        'combo_r': best_sw['combo_r'], 'marg_r': best_sw['marg_r'], 'mean_ll': best_sw['mean_ll'],
    },
}

with open('bayesian_walk_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to bayesian_walk_params.json')
print(json.dumps(output, indent=2))